In [1]:
from safetensors import safe_open
import torch
from torch import tensor
from datasets import load_dataset
from transformers import AutoTokenizer
import numpy as np
tokenizer = AutoTokenizer.from_pretrained("/mnt/shared-storage-user/dllm-share/Models/Qwen3/Qwen3-30B-A3B")

def clip_long_string(string, max_length=2000):
    """Clip long string to a maximum length."""
    # assert max_length > 50, "max_length must be greater than 50"
    if not len(string) > max_length:
        return string
    target_len = max_length - len("\n\n...(truncated)\n\n")
    return (
        string[: target_len // 2]
        + "\n\n...(truncated)\n\n"
        + string[-target_len // 2 :]
    )

In [2]:
a = torch.load('rollout.pt', weights_only=False)
gen_batch_output, final_mask, sample_index = a['gen_batch_output'], a['final_mask'], a['sample_index']

/mnt/shared-storage-user/dllm-share/liudawei/verl/verl/__init__.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [10]:
assert gen_batch_output.batch.batch_size == final_mask.shape == sample_index.shape

In [102]:
(sample_index == 17).nonzero().ravel()

tensor([  17,  145,  273,  401,  529,  737,  865,  993, 1121, 1249, 1377, 1582,
        1698, 1814, 1930, 2046, 2162, 2348, 2457, 2566, 2675, 2784, 2893, 3069,
        3170, 3271, 3372, 3473, 3574, 3735, 3835])

In [129]:
idx =  -127


batch = gen_batch_output.batch[idx]
print(tokenizer.decode(batch['responses'][batch['response_mask'].bool()]))
print("="*20)
print(clip_long_string(tokenizer.decode(batch['input_ids'][batch['attention_mask'].bool()]).split('<|im_start|>assistant')[0], max_length=3000))

\boxed{Benito Mussolini}<|im_end|>
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
You are presented with a problem and a previous memory. Please answer the problem based on the previous memory and put the answer in \boxed{{}}.

<problem>
Musician and satirist Allie Goertz wrote a song about the "The Simpsons" character Milhouse, who Matt Groening named after who?
</problem>

<memory>
Consolidated memory:
- Allie Goertz wrote a song about the "The Simpsons" character Milhouse.
- Matt Groening named the character Milhouse and gave him the middle name "Mussolini."
</memory>

Your answer:
<|im_end|>



In [4]:
ds = load_dataset('parquet', data_files='/mnt/shared-storage-user/dllm-share/liudawei/verl/data/hotpotqa/hotpotqa_train_32k.parquet')['train']

In [6]:
prompt_key = 'prompt'
context_key = 'context'
def doc2len(doc) -> int:
    messages = doc[prompt_key]
    assert all(isinstance(msg, dict) for msg in messages), f"{prompt_key} must contain dict messages"
    assert all("role" in msg and "content" in msg for msg in messages), f"each message in {prompt_key} must contain role/content"

    if context_key is not None:
        # [MemAgent] 对于 memagent 数据需要加上 context 部分的长度
        context = doc[context_key]
        assert isinstance(context, str), f"{context_key} must be str, got {type(context)}"

        user_idx = next((i for i, msg in enumerate(messages) if msg.get("role") == "user"), None)
        assert user_idx is not None, f"{prompt_key} must contain at least one user message"
        # Keep filtering logic read-only to avoid mutating source rows in multiprocessing.
        messages = [
            {**msg, "content": f"{msg['content']}\n\n{context}"} if i == user_idx else msg
            for i, msg in enumerate(messages)
        ]

    return len(tokenizer.apply_chat_template(messages, add_generation_prompt=True))


max_prompt_length = 32000
filter_desc = f"Filtering prompts longer than {max_prompt_length} tokens"

filter_fn = lambda doc: doc2len(doc) <= max_prompt_length  # 不要传 self, 无法序列化
def prompt_len(doc):
    return {"prompt_len": doc2len(doc)}
ds = ds.map(prompt_len, num_proc=64, desc="Calculating prompt lengths")

# ds1 = ds.filter(
#     filter_fn,
#     num_proc=32,
#     desc=filter_desc,
# )
# len(ds1)

Calculating prompt lengths (num_proc=64):   0%|          | 0/32768 [00:00<?, ? examples/s]

In [12]:
plen = np.array(ds['prompt_len'])

In [ ]:

path = "/mnt/shared-storage-user/dllm-share/liudawei/verl/checkpoints/qwen3_30b_moe_fsdp/global_step_1/actor/huggingface/model-00003-of-00025.safetensors"

with safe_open(path, framework="pt") as f:
    keys = list(f.keys())
    first_key = keys[0]
    print("first key:", first_key)

    tensor = f.get_tensor(first_key)
    print("dtype:", tensor.dtype)
    print("shape:", tensor.shape)

first key: model.layers.3.input_layernorm.weight
dtype: torch.float32
shape: torch.Size([2048])


下面是我的verl moe训练配置：(注意max_prompt_length, max_response_length不能更改)
```
    data.max_prompt_length=30000 \
    data.max_response_length=10000 \    
    actor_rollout_ref.model.use_remove_padding=True \
    actor_rollout_ref.actor.use_dynamic_bsz=True \
    actor_rollout_ref.ref.log_prob_use_dynamic_bsz=True \
    actor_rollout_ref.rollout.log_prob_use_dynamic_bsz=True \
    actor_rollout_ref.actor.ppo_max_token_len_per_gpu=10000 \
    actor_rollout_ref.ref.log_prob_max_token_len_per_gpu=10000 \
    actor_rollout_ref.rollout.log_prob_max_token_len_per_gpu=10000 \
    actor_rollout_ref.actor.ulysses_sequence_parallel_size=4 \
    actor_rollout_ref.ref.ulysses_sequence_parallel_size=4 \
```

我发现power没有打满 200W/700W, 显存也是 40GB/140GB，是不是可以增大类似于ppo_max_token_len_per_gpu这种参数到20k，允许更高的动态bsz跨样本？
首先 ppo_max_token_len_per_gpu 不能小于10k，因为总长度40k固定，sp=4，每张卡至少10k, 那么是不是可以再增加，达到我上面的目的呢？相当于增大了micro bsz，只不过我这里是动态bsz
或者减少sp size也可以？

In [15]:
from datasets import load_dataset
import torch

a = load_dataset("parquet", data_files="./data/hotpotqa/hotpotqa_train_32k.parquet", split='train')
a = a.shuffle(seed=1).select(range(20))

In [17]:
len(a[0]['context'])

101913

In [23]:
import random
def mapping(ex):
    length = random.randint(101913//2, 101913)
    ex['context'] = ex['context'][:length]
    return ex

b = a.map(mapping)

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [26]:
b.to_parquet("/mnt/shared-storage-user/dllm-share/liudawei/verl/data/test.parquet")

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

1479273

In [19]:
# 第一次采样
local_sample_index = tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15,  0,  1,
         2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15,  0,  1,  2,  3,
         4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15,  0,  1,  2,  3,  4,  5,
         6,  7,  8,  9, 10, 11, 12, 13, 14, 15,  0,  1,  2,  3,  4,  5,  6,  7,
         8,  9, 10, 11, 12, 13, 14, 15,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9,
        10, 11, 12, 13, 14, 15,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11,
        12, 13, 14, 15,  0,  1,  2,  3,  8,  9, 10, 11, 12, 13, 14, 15,  0,  1,
         2,  3,  8,  9, 10, 11,  0,  1,  2,  3,  8,  9, 10, 11,  0,  1,  2,  3,
         8,  9, 10, 11,  0,  1,  2,  3,  8,  9, 10, 11,  8,  9, 10, 11,  0,  1,
         2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15])
print(f"{local_sample_index.shape=}")
'''
local_reward_tensor = None
local_reward_tensor.max(1): max(
values=tensor([0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0.]),
indices=tensor([0, 0, 0, 0, 6, 0, 6, 0, 0, 0, 0, 7, 0, 0, 0, 0]))
'''

# 保留 4,5,6,7; 8,9,10,11

print((local_sample_index==4).sum())
print((local_sample_index==10).sum())
uid = '7d1c1982-c58c-4308-8030-6f420e49f62f', '904588be-ed4a-4417-8387-6351a911532d',


kept_reward_idxs=[4, 5, 6, 7, 8, 9, 10, 11]

'''
filtered_reward_tensor.max(1)
torch.return_types.max(
values=tensor([1., 0., 1., 0., 0., 0., 0., 1.]),
indices=tensor([6, 0, 6, 0, 0, 0, 0, 7]))
'''

'''
filtered_train_batch.non_tensor_batch['uid'][::4]
array(['7d1c1982-c58c-4308-8030-6f420e49f62f',
       '904588be-ed4a-4417-8387-6351a911532d',
       '7d1c1982-c58c-4308-8030-6f420e49f62f',
       '904588be-ed4a-4417-8387-6351a911532d',
       '7d1c1982-c58c-4308-8030-6f420e49f62f',
       '904588be-ed4a-4417-8387-6351a911532d',
       '7d1c1982-c58c-4308-8030-6f420e49f62f',
       '904588be-ed4a-4417-8387-6351a911532d',
       '7d1c1982-c58c-4308-8030-6f420e49f62f',
       '904588be-ed4a-4417-8387-6351a911532d',
       '7d1c1982-c58c-4308-8030-6f420e49f62f',
       '904588be-ed4a-4417-8387-6351a911532d',
       '7d1c1982-c58c-4308-8030-6f420e49f62f',
       '904588be-ed4a-4417-8387-6351a911532d',
       '904588be-ed4a-4417-8387-6351a911532d',
       '904588be-ed4a-4417-8387-6351a911532d',
       '904588be-ed4a-4417-8387-6351a911532d',
       '904588be-ed4a-4417-8387-6351a911532d',
       '904588be-ed4a-4417-8387-6351a911532d',
       '904588be-ed4a-4417-8387-6351a911532d',
       '7d1c1982-c58c-4308-8030-6f420e49f62f',
       '904588be-ed4a-4417-8387-6351a911532d'], dtype=object)
'''

local_sample_index.shape=torch.Size([176])
tensor(8)
tensor(14)


"\nfiltered_train_batch.non_tensor_batch['uid'][::4]\narray(['7d1c1982-c58c-4308-8030-6f420e49f62f',\n       '904588be-ed4a-4417-8387-6351a911532d',\n       '7d1c1982-c58c-4308-8030-6f420e49f62f',\n       '904588be-ed4a-4417-8387-6351a911532d',\n       '7d1c1982-c58c-4308-8030-6f420e49f62f',\n       '904588be-ed4a-4417-8387-6351a911532d',\n       '7d1c1982-c58c-4308-8030-6f420e49f62f',\n       '904588be-ed4a-4417-8387-6351a911532d',\n       '7d1c1982-c58c-4308-8030-6f420e49f62f',\n       '904588be-ed4a-4417-8387-6351a911532d',\n       '7d1c1982-c58c-4308-8030-6f420e49f62f',\n       '904588be-ed4a-4417-8387-6351a911532d',\n       '7d1c1982-c58c-4308-8030-6f420e49f62f',\n       '904588be-ed4a-4417-8387-6351a911532d',\n       '904588be-ed4a-4417-8387-6351a911532d',\n       '904588be-ed4a-4417-8387-6351a911532d',\n       '904588be-ed4a-4417-8387-6351a911532d',\n       '904588be-ed4a-4417-8387-6351a911532d',\n       '904588be-ed4a-4417-8387-6351a911532d',\n       '904588be-ed4a-4417-8387-63

In [20]:
# 第二次采样
local_sample_index = tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15,  0,  1,
         2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15,  0,  1,  2,  3,
         4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15,  0,  1,  2,  3,  4,  5,
         6,  7,  8,  9, 10, 11, 12, 13, 14, 15,  0,  1,  2,  3,  4,  5,  6,  7,
         8,  9, 10, 11, 12, 13, 14, 15,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9,
        10, 11, 12, 13, 14, 15,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11,
        12, 13, 14, 15,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13,
        14, 15,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 12, 13, 14, 15,
        12, 13, 14, 15,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13,
        14, 15])
print(f"{local_sample_index.shape=}")
'''
这里我强行赋值
local_reward_tensor.max(1)
torch.return_types.max(
values=tensor([1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.]),
indices=tensor([1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]))
'''

# 保留 0, 1, 2, 3; 8, 9, 10, 11

print((local_sample_index==1).sum())
print((local_sample_index==8).sum())
uid = '7815e95e-a2e8-4134-b1f2-7732c3df874a', '8617d7e6-958b-492c-8d73-bb2e954368e6',


kept_reward_idxs=[0, 1, 2, 3, 8, 9, 10, 11]

'''
filtered_reward_tensor.max(1)
torch.return_types.max(
values=tensor([1., 1., 1., 0., 0., 0., 0., 1.]),
indices=tensor([1, 1, 1, 0, 0, 0, 0, 1]))
'''

'''

filtered_train_batch.non_tensor_batch['uid'][::4]
array(['7815e95e-a2e8-4134-b1f2-7732c3df874a',
       '8617d7e6-958b-492c-8d73-bb2e954368e6',
       '7815e95e-a2e8-4134-b1f2-7732c3df874a',
       '8617d7e6-958b-492c-8d73-bb2e954368e6',
       '7815e95e-a2e8-4134-b1f2-7732c3df874a',
       '8617d7e6-958b-492c-8d73-bb2e954368e6',
       '7815e95e-a2e8-4134-b1f2-7732c3df874a',
       '8617d7e6-958b-492c-8d73-bb2e954368e6',
       '7815e95e-a2e8-4134-b1f2-7732c3df874a',
       '8617d7e6-958b-492c-8d73-bb2e954368e6',
       '7815e95e-a2e8-4134-b1f2-7732c3df874a',
       '8617d7e6-958b-492c-8d73-bb2e954368e6',
       '7815e95e-a2e8-4134-b1f2-7732c3df874a',
       '8617d7e6-958b-492c-8d73-bb2e954368e6',
       '7815e95e-a2e8-4134-b1f2-7732c3df874a',
       '8617d7e6-958b-492c-8d73-bb2e954368e6',
       '8617d7e6-958b-492c-8d73-bb2e954368e6',
       '7815e95e-a2e8-4134-b1f2-7732c3df874a',
       '8617d7e6-958b-492c-8d73-bb2e954368e6'], dtype=object)
'''

local_sample_index.shape=torch.Size([164])
tensor(9)
tensor(10)


"\n\nfiltered_train_batch.non_tensor_batch['uid'][::4]\narray(['7815e95e-a2e8-4134-b1f2-7732c3df874a',\n       '8617d7e6-958b-492c-8d73-bb2e954368e6',\n       '7815e95e-a2e8-4134-b1f2-7732c3df874a',\n       '8617d7e6-958b-492c-8d73-bb2e954368e6',\n       '7815e95e-a2e8-4134-b1f2-7732c3df874a',\n       '8617d7e6-958b-492c-8d73-bb2e954368e6',\n       '7815e95e-a2e8-4134-b1f2-7732c3df874a',\n       '8617d7e6-958b-492c-8d73-bb2e954368e6',\n       '7815e95e-a2e8-4134-b1f2-7732c3df874a',\n       '8617d7e6-958b-492c-8d73-bb2e954368e6',\n       '7815e95e-a2e8-4134-b1f2-7732c3df874a',\n       '8617d7e6-958b-492c-8d73-bb2e954368e6',\n       '7815e95e-a2e8-4134-b1f2-7732c3df874a',\n       '8617d7e6-958b-492c-8d73-bb2e954368e6',\n       '7815e95e-a2e8-4134-b1f2-7732c3df874a',\n       '8617d7e6-958b-492c-8d73-bb2e954368e6',\n       '8617d7e6-958b-492c-8d73-bb2e954368e6',\n       '7815e95e-a2e8-4134-b1f2-7732c3df874a',\n       '8617d7e6-958b-492c-8d73-bb2e954368e6'], dtype=object)\n"

In [ ]:
'''最终过滤: 164 -> 124   (8+14 + 9+10) *4 -> (8+14 +9) *4
4 = rollout.n, 前面是turns数量
'''


reward_tensor_batch.max(1)
torch.return_types.max(
values=tensor([1., 0., 1., 0., 0., 0., 0., 1., 1., 1., 1., 0.]),
indices=tensor([6, 0, 6, 0, 0, 0, 0, 7, 1, 1, 1, 0]))


advantage_scalar=\
tensor([ 0.5000, -0.5000,  0.5000, -0.5000, -0.2500, -0.2500, -0.2500,  0.7500,
         0.2500,  0.2500,  0.2500, -0.7500])
advantage_scalar=\
tensor([ 0.5000, -0.5000,  0.5000, -0.5000, -0.2500, -0.2500, -0.2500,  0.7500,
         0.5000, -0.5000,  0.5000, -0.5000, -0.2500, -0.2500, -0.2500,  0.7500,
         0.5000, -0.5000,  0.5000, -0.5000, -0.2500, -0.2500, -0.2500,  0.7500,
         0.5000, -0.5000,  0.5000, -0.5000, -0.2500, -0.2500, -0.2500,  0.7500,
         0.5000, -0.5000,  0.5000, -0.5000, -0.2500, -0.2500, -0.2500,  0.7500,
         0.5000, -0.5000,  0.5000, -0.5000, -0.2500, -0.2500, -0.2500,  0.7500,
         0.5000, -0.5000,  0.5000, -0.5000, -0.2500, -0.2500, -0.2500,  0.7500,
        -0.2500, -0.2500, -0.2500,  0.7500, -0.2500, -0.2500, -0.2500,  0.7500,
        -0.2500, -0.2500, -0.2500,  0.7500, -0.2500, -0.2500, -0.2500,  0.7500,
        -0.2500, -0.2500, -0.2500,  0.7500, -0.2500, -0.2500, -0.2500,  0.7500,
         0.5000, -0.5000,  0.5000, -0.5000, -0.2500, -0.2500, -0.2500,  0.7500,
         0.2500,  0.2500,  0.2500, -0.7500,  0.2500,  0.2500,  0.2500, -0.7500,
         0.2500,  0.2500,  0.2500, -0.7500,  0.2500,  0.2500,  0.2500, -0.7500,
         0.2500,  0.2500,  0.2500, -0.7500,  0.2500,  0.2500,  0.2500, -0.7500,
         0.2500,  0.2500,  0.2500, -0.7500,  0.2500,  0.2500,  0.2500, -0.7500,
         0.2500,  0.2500,  0.2500, -0.7500])